# Mini-Project 02 — Will This Be a Strong Sales Week?
**DS5020 · Bayes' theorem and Naive Bayes on real Walmart sales data**

| Role | Name |
|---|---|
| Data Analyst | Muhammad Saad Malik |
| ML Engineer |  |

**How to run:** 
Cells marked `# TODO` are to be filled in. Every task writes its result into `answers`, and the last cell prints the answer sheet.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

pd.set_option("display.precision", 4)

# Works whether Jupyter was started in the repo root or in notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
DATA_PATH = ROOT / "data" / "Walmart.csv"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)
assert DATA_PATH.exists(), f"Walmart.csv not found in {DATA_PATH.parent} (see data/README.md)"

answers = {}   # task -> value; printed as the answer sheet at the end

## 1. Event definitions (R1, R2)

Written **before** any computation and used unchanged in every task.

**Sample space (R2):** one outcome = one store-week; $S$ (the sample space) = all 6,435 store-weeks.
*Notation clash:* the brief uses $S$ both for the sample space and for "strong week". Below, `S` always means **strong week**.

| Event | Definition (one store-week) |
|---|---|
| **S** strong week | `Weekly_Sales` ≥ 1.10 × that store's average weekly sales |
| **H** holiday week | `Holiday_Flag` = 1 |
| **Q** holiday season | the week's month is November or December |
| **C** cold week | `Temperature` < 40 °F |
| **F** expensive fuel | `Fuel_Price` ≥ \$3.50 |
| **J** high unemployment | `Unemployment` ≥ 8 % |
| **G** large store | that store's average weekly sales ≥ \$1,000,000 |

S and G depend on the store average. Parts P and B use the average over all 143 weeks; Part N uses the **training weeks only** (R3).

In [2]:
# --- R1: thresholds live here, once ---
STRONG_FACTOR  = 1.10        # S
SEASON_MONTHS  = [11, 12]    # Q
COLD_F         = 40          # C  (strictly below)
FUEL_HIGH      = 3.50        # F  (at least)
UNEMP_HIGH     = 8.0         # J  (at least)
LARGE_STORE    = 1_000_000   # G  (at least)

EVENTS   = list("SHQCFJG")
FEATURES = ["H", "Q", "C", "F", "J", "G"]    # Naive Bayes inputs; S is the target


def add_events(d, store_avg):
    # Add the seven boolean event columns.
    # store_avg: Series indexed by Store. Pass training-only averages in Part N (R3).
    d = d.copy()
    avg = d["Store"].map(store_avg)
    d["S"] = d["Weekly_Sales"] >= STRONG_FACTOR * avg
    d["H"] = d["Holiday_Flag"] == 1
    d["Q"] = d["Date"].dt.month.isin(SEASON_MONTHS)
    d["C"] = d["Temperature"] < COLD_F
    d["F"] = d["Fuel_Price"] >= FUEL_HIGH
    d["J"] = d["Unemployment"] >= UNEMP_HIGH
    d["G"] = avg >= LARGE_STORE
    return d


# Holiday weeks from Section 2 of the brief (needed for B3)
HOLIDAY_DATES = {
    "Super Bowl":   ["2010-02-12", "2011-02-11", "2012-02-10"],
    "Labor Day":    ["2010-09-10", "2011-09-09", "2012-09-07"],
    "Thanksgiving": ["2010-11-26", "2011-11-25"],
    "Christmas":    ["2010-12-31", "2011-12-30"],
}
DATE_TO_HOLIDAY = {pd.Timestamp(day): name for name, days in HOLIDAY_DATES.items() for day in days}
GROUPS = list(HOLIDAY_DATES) + ["No holiday"]

# print(DATE_TO_HOLIDAY)
print(GROUPS)

['Super Bowl', 'Labor Day', 'Thanksgiving', 'Christmas', 'No holiday']


### Helpers: probability by counting, and the R5 check
`A` and `B` can be a column name (`"S"`) or a boolean Series (`ev.S & ev.H`, `~ev.H`).

In [3]:
def _col(d, A):
    return d[A] if isinstance(A, str) else A

def P(d, A):
    # P(A) = #A / #rows
    return _col(d, A).mean()

def P_given(d, A, B):
    # P(A | B): count inside the rows where B is true
    a, b = _col(d, A), _col(d, B)
    assert b.sum() > 0, "conditioning event is empty"
    return a[b].mean()

def check(x, y, label="", tol=1e-9):
    # R5: two ways of computing the same probability must agree
    assert abs(x - y) < tol, f"{label}: {x} vs {y}"
    print(f"OK  {label}: {x:.4f} = {y:.4f}")

def r4(x):
    # plain rounded numbers for the answer sheet
    if isinstance(x, (tuple, list)): return tuple(r4(v) for v in x)
    if isinstance(x, dict):          return {k: r4(v) for k, v in x.items()}
    return round(float(x), 4) if isinstance(x, (float, np.floating)) else x


### WARM UP part will be submitted seperately

## Part P — The probability of each event
### P1 · Load and check the data

In [4]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

n_rows, n_stores, n_weeks = len(df), df["Store"].nunique(), df["Date"].nunique()
n_hol = int(df["Holiday_Flag"].sum())
print(f"{n_rows} rows, {n_stores} stores, {n_weeks} weeks, {n_hol} holiday store-weeks")

assert (n_rows, n_stores, n_weeks, n_hol) == (6435, 45, 143, 450)
assert df.isna().sum().sum() == 0
assert df.groupby("Store").size().eq(143).all()          # every store has every week
assert (df["Date"].dt.day_of_week == 4).all()               # all Fridays -> day/month parsed correctly
assert set(df.loc[df.Holiday_Flag == 1, "Date"]) == set(DATE_TO_HOLIDAY)

answers["P1"] = f"{n_rows} rows, {n_stores} stores, {n_weeks} weeks, {n_hol} holiday store-weeks"
df.head()

6435 rows, 45 stores, 143 weeks, 450 holiday store-weeks


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,2010-02-05,1.6437e+06,0,42.31,2.572,211.0964,8.106
1,1,2010-02-12,1.6420e+06,1,38.51,2.548,211.2422,8.106
2,1,2010-02-19,1.6120e+06,0,39.93,2.514,211.2891,8.106
3,1,2010-02-26,1.4097e+06,0,46.63,2.561,211.3196,8.106
4,1,2010-03-05,1.5548e+06,0,46.50,2.625,211.3501,8.106


### P2 · Probability of each event, checked with the complement (R5)
Store averages over **all** weeks here (Parts P and B).

In [5]:
add_events(df, df.groupby("Store")["Weekly_Sales"].mean()).head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,S,H,Q,C,F,J,G
0,1,2010-02-05,1.6437e+06,0,42.31,2.572,211.0964,8.106,False,False,False,False,False,True,True
1,1,2010-02-12,1.6420e+06,1,38.51,2.548,211.2422,8.106,False,True,False,True,False,True,True
2,1,2010-02-19,1.6120e+06,0,39.93,2.514,211.2891,8.106,False,False,False,True,False,True,True
3,1,2010-02-26,1.4097e+06,0,46.63,2.561,211.3196,8.106,False,False,False,False,False,True,True
4,1,2010-03-05,1.5548e+06,0,46.50,2.625,211.3501,8.106,False,False,False,False,False,True,True


In [6]:
ev = add_events(df, df.groupby("Store")["Weekly_Sales"].mean())

event_table = pd.DataFrame({
    "#E":     [int(ev[e].sum()) for e in EVENTS],
    "#E^c":   [int((~ev[e]).sum()) for e in EVENTS],
    "P(E)":   [P(ev, e) for e in EVENTS],
    "P(E^c)": [P(ev, ~ev[e]) for e in EVENTS],
}, index=EVENTS)
event_table["P(E)+P(E^c)"] = event_table["P(E)"] + event_table["P(E^c)"]

assert (event_table["#E"] + event_table["#E^c"] == len(ev)).all()    # R5 by counting
assert np.allclose(event_table["P(E)+P(E^c)"], 1)                     # R5 by complement
check(P(ev, "H"), 450 / 6435, "P(H) = 450/6435")

answers["P2"] = event_table["P(E)"].round(4).to_dict()
event_table

OK  P(H) = 450/6435: 0.0699 = 0.0699


,#E,#E^c,P(E),P(E^c),P(E)+P(E^c)
S,814,5621,0.1265,0.8735,1.0
H,450,5985,0.0699,0.9301,1.0
Q,810,5625,0.1259,0.8741,1.0
C,989,5446,0.1537,0.8463,1.0
F,3029,3406,0.4707,0.5293,1.0
J,3022,3413,0.4696,0.5304,1.0
G,3146,3289,0.4889,0.5111,1.0


### P3 · Joint probabilities

In [7]:
# TODO P3: joint probabilities by counting, e.g. P(ev, ev.S & ev.H)
joint = {
    "P(S∩H)": P(ev, ev.S & ev.H),
    "P(S∩Q)": P(ev, ev.S & ev.Q),
    "P(H∩Q)": P(ev, ev.H & ev.Q),
    "P(S∩C)": P(ev, ev.S & ev.C)
}
# TODO R5: check each one a second way (e.g. count / 6435, or P(A|B)·P(B))
pairs = {"P(S∩H)": ("S", "H"), "P(S∩Q)": ("S", "Q"), "P(H∩Q)": ("H", "Q"), "P(S∩C)": ("S", "C")}
for name, (a, b) in pairs.items():
    check(joint[name], P_given(ev, a, b) * P(ev, b), f"{name}: counting vs P({a}|{b})P({b})")

answers["P3"] = joint
joint

OK  P(S∩H): counting vs P(S|H)P(H): 0.0190 = 0.0190
OK  P(S∩Q): counting vs P(S|Q)P(Q): 0.0517 = 0.0517
OK  P(H∩Q): counting vs P(H|Q)P(Q): 0.0280 = 0.0280
OK  P(S∩C): counting vs P(S|C)P(C): 0.0287 = 0.0287


{'P(S∩H)': np.float64(0.01895881895881896),
 'P(S∩Q)': np.float64(0.05174825174825175),
 'P(H∩Q)': np.float64(0.027972027972027972),
 'P(S∩C)': np.float64(0.028749028749028748)}

## Part B — Conditional probability and Bayes' theorem
### B1 · Holidays

In [8]:
# TODO B1: count inside each group
p_S_H  = P_given(ev, ev.S, ev.H)     # P(S | H)
p_S_Hc = P_given(ev, ev.S, ~ev.H)    # P(S | H^c)

answers["B1"] = (round(float(p_S_H), 4), round(float(p_S_Hc), 4))
print(answers["B1"])

(0.2711, 0.1156)


_B1 sentence:_ In a holiday week a store has a strong week 27.1% of the time, against 11.6% in other weeks, so a holiday makes a strong week about 2.3 times as likely.

### B2 · Reverse it with Bayes
$$P(H \mid S) = \frac{P(S \mid H)\,P(H)}{P(S)}$$

In [9]:
# TODO B2
p_H_S_bayes = (p_S_H* P(ev, ev.H))/P(ev, ev.S)  # from p_S_H, P(H), P(S)
p_H_S_count = P_given(ev, ev.H, ev.S)   # count inside the strong weeks
check(p_H_S_bayes, p_H_S_count, "P(H|S): Bayes vs counting")


answers["B2"] = (p_H_S_bayes, p_H_S_count)

OK  P(H|S): Bayes vs counting: 0.1499 = 0.1499


### B3 · Total probability over the five holiday groups
$$P(S) = \sum_g P(S \mid g)\,P(g)$$

In [10]:
ev["group"] = ev["Date"].map(DATE_TO_HOLIDAY).fillna("No holiday")
# print(ev.head())
assert (ev["group"] != "No holiday").sum() == 450

b3 = pd.DataFrame(index=GROUPS, columns=["n", "P(g)", "P(S|g)", "P(S|g)P(g)", "P(g|S)"], dtype=float)
# TODO B3: fill one row per group g (loop over GROUPS; ev["group"] == g is the event g)
b3["n"] = ev['group'].value_counts()
b3["P(g)"] = b3["n"] / b3["n"].sum()
# print(b3["P(g)"].sum())
b3["P(S|g)"] = ev.groupby('group')['S'].mean()
b3["P(S|g)P(g)"] = b3["P(S|g)"]*b3["P(g)"]
b3["P(g|S)"] = b3["P(S|g)P(g)"]/P(ev, ev.S)



# TODO R5: b3["P(g)"].sum() == 1,  b3["P(S|g)P(g)"].sum() == P(ev, "S"),  b3["P(g|S)"].sum() == 1

check(b3["P(g)"].sum(), 1, "B3: P(g) sums to 1")
check(b3["P(S|g)P(g)"].sum(), P(ev, "S"), "B3: total probability rebuilds P(S)")
check(b3["P(g|S)"].sum(), 1, "B3: P(g|S) sums to 1")

answers["B3"] = (b3.drop("No holiday")["P(S|g)"].idxmax(),   # holiday with highest P(S|g)
                 b3["P(g|S)"].idxmax())                       # group with largest P(g|S)

OK  B3: P(g) sums to 1: 1.0000 = 1.0000
OK  B3: total probability rebuilds P(S): 0.1265 = 0.1265
OK  B3: P(g|S) sums to 1: 1.0000 = 1.0000


_B3 sentence:_ Thanksgiving has by far the highest chance of a strong week (P(S | g) = 0.82), but non-holiday weeks supply most strong weeks (P(No holiday | S) = 0.85) because they are 93% of all weeks. The Christmas-flagged week has the lowest rate (0.07): in this data it is the week ending 30/31 December, after Christmas Day, so the pre-Christmas rush falls in a week that isn’t flagged as a holiday.

### B4 · Season

In [11]:
# TODO B4
p_S_Q = P_given(ev, ev.S, ev.Q)   # P(S | Q)
p_Q_S = P_given(ev, ev.Q, ev.S)   # P(Q | S)

answers["B4"] = (p_S_Q, p_Q_S)
print(answers["B4"])

(np.float64(0.4111111111111111), np.float64(0.4090909090909091))


_B4 — why are they different?_ …

P(S | Q) = 0.411 and P(Q | S) = 0.409. P(S | Q) says that 41% of Nov–Dec store-weeks are strong, while P(Q | S) says that 41% of all strong weeks fall in Nov–Dec; they measure different things because one divides by the number of Nov–Dec weeks and the other by the number of strong weeks. By Bayes’ theorem, P(Q | S) = P(S | Q)·P(Q)/P(S), and the two are nearly equal here only because Nov–Dec weeks (810) and strong weeks (814) are about equally common.


### B5 · The staffing rules: precision and recall
For a rule "extra staff when event $A$":

- precision $= P(S \mid A) = \dfrac{P(A \mid S)\,P(S)}{P(A \mid S)\,P(S) + P(A \mid S^c)\,P(S^c)}$ (Bayes + total probability), checked by counting
- recall $= P(A \mid S)$ = the fraction of strong weeks that get extra staff

In [12]:
def precision_by_bayes(p_A_given_S, p_A_given_Sc, p_S):
    # TODO B5: return P(S | A) using the formula above
    p_S_given_A = (p_A_given_S*p_S)/((p_A_given_S * p_S)+(p_A_given_Sc * (1-p_S)))
    return p_S_given_A


rules = pd.DataFrame(index=["holiday (H)", "Nov–Dec (Q)"],
                     columns=["P(A|S)", "P(A|S^c)", "precision (Bayes)", "precision (count)", "recall", "missed"],
                     dtype=float)
# TODO B5: fill both rows; "missed" = 1 - recall. Check Bayes vs counting (R5).
#print(P_given(ev, ev.H, ev.S))
rules.loc["holiday (H)", "P(A|S)"] = P_given(ev, ev.H, ev.S)
rules.loc["Nov–Dec (Q)", "P(A|S)"] = P_given(ev, ev.Q, ev.S)
rules.loc["holiday (H)", "P(A|S^c)"] = P_given(ev, ev.H, ~ev.S)
rules.loc["Nov–Dec (Q)", "P(A|S^c)"] = P_given(ev, ev.Q, ~ev.S)
 # P(S|H) = P(H|S)*P(S)/( P(H|S)*P(S)+P(H|S^c)*P(S^c))     bayes
rules.loc["holiday (H)", "precision (Bayes)"] =  precision_by_bayes(rules.loc["holiday (H)", "P(A|S)"],rules.loc["holiday (H)", "P(A|S^c)"],P(ev, ev.S))
rules.loc["Nov–Dec (Q)", "precision (Bayes)"] =  precision_by_bayes(rules.loc["Nov–Dec (Q)", "P(A|S)"], rules.loc["Nov–Dec (Q)", "P(A|S^c)"], P(ev, ev.S))
rules.loc["holiday (H)", "precision (count)"] = P_given(ev, ev.S, ev.H)
rules.loc["Nov–Dec (Q)", "precision (count)"] = P_given(ev, ev.S, ev.Q)
rules.loc["holiday (H)", "recall"] = rules.loc["holiday (H)", "P(A|S)"]
rules.loc["Nov–Dec (Q)", "recall"] = rules.loc["Nov–Dec (Q)", "P(A|S)"]
rules.loc["holiday (H)", "missed"] = 1 - rules.loc["holiday (H)", "recall"]
rules.loc["Nov–Dec (Q)", "missed"] = 1 - rules.loc["Nov–Dec (Q)", "recall"]

for rule in rules.index:
    check(rules.loc[rule, "precision (Bayes)"], rules.loc[rule, "precision (count)"],
          f"{rule} precision: Bayes vs counting")

answers["B5"] = {
    rule: (round(float(rules.loc[rule, "precision (count)"]), 4),   # precision
           round(float(rules.loc[rule, "recall"]), 4))              # recall
    for rule in rules.index
}
rules


OK  holiday (H) precision: Bayes vs counting: 0.2711 = 0.2711
OK  Nov–Dec (Q) precision: Bayes vs counting: 0.4111 = 0.4111


,P(A|S),P(A|S^c),precision (Bayes),precision (count),recall,missed
holiday (H),0.1499,0.0584,0.2711,0.2711,0.1499,0.8501
Nov–Dec (Q),0.4091,0.0849,0.4111,0.4111,0.4091,0.5909


_B5:_ The holiday rule has precision 0.27 and recall 0.15: only 27% of holiday weeks are strong, and it misses 85% of all strong weeks. The Nov–Dec rule is better on both counts (precision 0.41, recall 0.41), though it still misses 59% of strong weeks.

### B6 · Base rate
Keep $P(H \mid S)$ and $P(H \mid S^c)$ from B5 fixed and change only $P(S)$.

In [13]:
# TODO B6: precision_by_bayes(..., p_S = 2 * P(S)) and (..., p_S = P(S) / 2)
a_S  = rules.loc["holiday (H)", "P(A|S)"]
a_Sc = rules.loc["holiday (H)", "P(A|S^c)"]
p_S  = P(ev, "S")

prec_doubled = precision_by_bayes(a_S, a_Sc, p_S*2)
prec_halved  = precision_by_bayes(a_S, a_Sc, p_S*0.5)

def p_Sc_given_A(a_S, a_Sc, p):
    # P(S^c | A) by Bayes, computed separately from precision_by_bayes
    return a_Sc * (1 - p) / (a_S * p + a_Sc * (1 - p))

for label, p in [("doubled", 2 * p_S), ("halved", p_S / 2)]:
    check(precision_by_bayes(a_S, a_Sc, p) + p_Sc_given_A(a_S, a_Sc, p), 1,
          f"B6 {label}: P(S|H) + P(S^c|H) = 1")

answers["B6"] = (round(float(prec_doubled),4), round(float(prec_halved),4))
answers["B6"]


OK  B6 doubled: P(S|H) + P(S^c|H) = 1: 1.0000 = 1.0000
OK  B6 halved: P(S|H) + P(S^c|H) = 1: 1.0000 = 1.0000


(0.4652, 0.1478)

_B6_: With the same rule, precision would be 0.465 if strong weeks were twice as common and 0.148 if they were half as common (it is 0.271 now), so how trustworthy the holiday rule is depends on the base rate.

NOTE: B6 uses hypothetical base rates, so it can’t be checked by counting. The function precision_by_bayes was checked against counting in B5 (0.2711 = 0.2711), and here each scenario is checked by summing P(S | H) + P(Sᶜ | H) = 1.

### B7 · Independence checks
If $A$ and $B$ are independent, $P(A \cap B) = P(A)\,P(B)$. The ratio $P(S\cap B)\,/\,[P(S)P(B)] = P(S\mid B)/P(S)$ shows how far from independence each pair is (1 = independent).

In [14]:
b7 = pd.DataFrame(index=[f"(S, {b})" for b in FEATURES],
                  columns=["P(S∩B)", "P(S)P(B)", "ratio"], dtype=float)
# TODO B7: fill the table

for b in FEATURES:
    b7.loc[f"(S, {b})", "P(S∩B)"] = P(ev, ev.S & ev[b])
    b7.loc[f"(S, {b})",  "P(S)P(B)"] = P(ev, ev.S) * P(ev, ev[b])
    b7.loc[f"(S, {b})",  "ratio"] = b7.loc[f"(S, {b})", "P(S∩B)"] / b7.loc[f"(S, {b})", "P(S)P(B)"]
    check(b7.loc[f"(S, {b})", "P(S∩B)"], P_given(ev, "S", b) * P(ev, b), f"P(S∩{b}): counting vs P(S|{b})P({b})")

b7

OK  P(S∩H): counting vs P(S|H)P(H): 0.0190 = 0.0190
OK  P(S∩Q): counting vs P(S|Q)P(Q): 0.0517 = 0.0517
OK  P(S∩C): counting vs P(S|C)P(C): 0.0287 = 0.0287
OK  P(S∩F): counting vs P(S|F)P(F): 0.0452 = 0.0452
OK  P(S∩J): counting vs P(S|J)P(J): 0.0640 = 0.0640
OK  P(S∩G): counting vs P(S|G)P(G): 0.0570 = 0.0570


,P(S∩B),P(S)P(B),ratio
"(S, H)",0.0190,0.0088,2.1432
"(S, Q)",0.0517,0.0159,3.2500
"(S, C)",0.0287,0.0194,1.4788
"(S, F)",0.0452,0.0595,0.7595
"(S, J)",0.0640,0.0594,1.0778
"(S, G)",0.0570,0.0618,0.9222


_B7 sentence — which events carry information about strong weeks?_ …

Q, H and C carry the most information: they make a strong week more likely (ratio 3.25, 2.14 and 1.48, so Nov–Dec weeks are 3.25 times as likely to be strong), F makes it less likely (0.76), while J and G are close to independent of S (ratios 1.08 and 0.92) and say little about strong weeks.

In [15]:
pd.Series({k: r4(v) for k, v in answers.items()}, name="value").to_frame()

,value
P1,"6435 rows, 45 stores, 143 weeks, 450 holiday s..."
P2,"{'S': 0.1265, 'H': 0.0699, 'Q': 0.1259, 'C': 0..."
P3,"{'P(S∩H)': 0.019, 'P(S∩Q)': 0.0517, 'P(H∩Q)': ..."
B1,"(0.2711, 0.1156)"
B2,"(0.1499, 0.1499)"
B3,"(Thanksgiving, No holiday)"
B4,"(0.4111, 0.4091)"
B5,"{'holiday (H)': ('0.2711', '0.1499'), 'Nov–Dec..."
B6,"(0.4652, 0.1478)"


## Part N — A Naive Bayes classifier for strong weeks
### N1 · Split by time (R3)
Train on 2010–2011, test on 2012. **Store averages come from the training weeks only**, so S and G in the test set use no 2012 information.

In [16]:
train_raw = df[df["Date"].dt.year <= 2011]
test_raw  = df[df["Date"].dt.year == 2012]
assert len(train_raw) + len(test_raw) == len(df)

store_avg_train = train_raw.groupby("Store")["Weekly_Sales"].mean()   # R3
train = add_events(train_raw, store_avg_train)
test  = add_events(test_raw,  store_avg_train)
print(f"train: {len(train)} store-weeks ({train.Date.nunique()} weeks)   "
      f"test: {len(test)} store-weeks ({test.Date.nunique()} weeks)")

# TODO N1: report P(S) in each set
p_S_train = None
p_S_test  = None

train: 4500 store-weeks (100 weeks)   test: 1935 store-weeks (43 weeks)


### N2 · Train by counting (training set only)

In [17]:
# TODO N2: priors
prior_S, prior_Sc = None, None

# TODO N2: likelihoods P(x=1 | S) and P(x=1 | S^c) for each feature
lik = pd.DataFrame(index=FEATURES,
                   columns=["P(x=1|S)", "P(x=1|S^c)", "P(x=1|S) smoothed", "P(x=1|S^c) smoothed"],
                   dtype=float)
# Note for N5: CategoricalNB(alpha=1) uses Laplace smoothing. For a 0/1 feature that is
#   (count of x=1 in the class + 1) / (class size + 2).
# The priors are NOT smoothed. Keep both versions so N3 can match N5 exactly.

lik

,P(x=1|S),P(x=1|S^c),P(x=1|S) smoothed,P(x=1|S^c) smoothed
H,NaN,NaN,NaN,NaN
Q,NaN,NaN,NaN,NaN
C,NaN,NaN,NaN,NaN
F,NaN,NaN,NaN,NaN
J,NaN,NaN,NaN,NaN
G,NaN,NaN,NaN,NaN


### N3 · Predict by hand
$$\text{score}(S) = P(S)\prod_i P(x_i \mid S), \qquad \text{score}(S^c) = P(S^c)\prod_i P(x_i \mid S^c), \qquad P(S \mid x) = \frac{\text{score}(S)}{\text{score}(S) + \text{score}(S^c)}$$
For a feature with $x_i = 0$, use $P(x_i = 0 \mid S) = 1 - P(x_i = 1 \mid S)$.

In [18]:
profiles = pd.DataFrame(
    [[1, 1, 1, 0, 0, 1],    # (a) holiday, Nov–Dec, cold, cheap fuel, low unemployment, large store
     [0, 0, 0, 1, 1, 0]],   # (b) no holiday, not Nov–Dec, not cold, expensive fuel, high unemployment, small store
    columns=FEATURES, index=["a", "b"])

# TODO N3: the two scores and the normalization for each profile (show your steps)
post_a = None
post_b = None

answers["N3"] = (post_a, post_b)
profiles

,H,Q,C,F,J,G
a,1,1,1,0,0,1
b,0,0,0,1,1,0


### N4 · Check the naive assumption for (H, Q)

In [19]:
# TODO N4: on the training set, compare P(H∩Q | S) with P(H|S)·P(Q|S)
lhs = None   # P(H∩Q | S)
rhs = None   # P(H | S) * P(Q | S)

_N4 — why are H and Q not independent?_ …

### N5 · scikit-learn

In [20]:
X_train, y_train = train[FEATURES].astype(int), train["S"].astype(int)
X_test,  y_test  = test[FEATURES].astype(int),  test["S"].astype(int)

nb = CategoricalNB(alpha=1, min_categories=2).fit(X_train, y_train)
sk_post = nb.predict_proba(profiles)[:, 1]      # column 1 = class S
print("scikit-learn P(S | a), P(S | b):", sk_post.round(4))

# TODO N5: check(sk_post[0], post_a, "profile a: hand vs sklearn"), same for b

scikit-learn P(S | a), P(S | b): [0.7939 0.0301]


### N6 · Evaluate on the test set (2012)

In [21]:
def evaluate(y, yhat):
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "accuracy": accuracy_score(y, yhat),
            # zero_division=0: "never strong" predicts no S, so its precision is 0/0. Say so in words.
            "precision P(S|Ŝ)": precision_score(y, yhat, zero_division=0),
            "recall P(Ŝ|S)": recall_score(y, yhat, zero_division=0)}

preds = {
    "Naive Bayes":       None,                                  # TODO N6: nb.predict(...)
    "never strong":      np.zeros(len(test), dtype=int),
    "strong if holiday": test["H"].astype(int).to_numpy(),
}
results = pd.DataFrame({name: evaluate(y_test, yhat) for name, yhat in preds.items() if yhat is not None}).T

answers["N6"] = results[["accuracy", "precision P(S|Ŝ)", "recall P(Ŝ|S)"]].round(4).to_dict("index")
results

,TP,FP,FN,TN,accuracy,precision P(S|Ŝ),recall P(Ŝ|S)
never strong,0.0,0.0,255.0,1680.0,0.8682,0.0000,0.0000
strong if holiday,23.0,67.0,232.0,1613.0,0.8455,0.2556,0.0902


_N6 — the test year has no Thanksgiving or Christmas. How does that affect the results?_ …

### N7 · Threshold (stretch)

In [22]:
p_test = nb.predict_proba(X_test)[:, 1]     # P(S | x) for every test store-week

# TODO N7: for t in [0.1, 0.2, 0.3, 0.5]: predict S when p_test > t, then precision and recall
n7 = pd.DataFrame(index=[0.1, 0.2, 0.3, 0.5], columns=["precision", "recall", "# predicted S"], dtype=float)
n7

,precision,recall,# predicted S
0.1,NaN,NaN,NaN
0.2,NaN,NaN,NaN
0.3,NaN,NaN,NaN
0.5,NaN,NaN,NaN


## J1 · Note to Maria
Write the five sentences in `submission/note.txt`: how often stores have a strong week; which holiday and season matter most; the honest truth about the holiday rule (precision and what it misses); whether the model beats the rule; one risk. Store numbers only (R4).

## Answer sheet
Copy these into `submission/answer_sheet.md`.

In [23]:
pd.Series(answers, name="value").to_frame()

,value
P1,"6435 rows, 45 stores, 143 weeks, 450 holiday s..."
P2,"{'S': 0.1265, 'H': 0.0699, 'Q': 0.1259, 'C': 0..."
P3,"{'P(S∩H)': 0.01895881895881896, 'P(S∩Q)': 0.05..."
B1,"(0.2711, 0.1156)"
B2,"(0.1499, 0.1499)"
B3,"(Thanksgiving, No holiday)"
B4,"(0.4111, 0.4091)"
B5,"{'holiday (H)': ('0.2711', '0.1499'), 'Nov–Dec..."
B6,"(0.4652, 0.1478)"
N3,"(None, None)"
